In [13]:
#import packages
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import warnings

cases_df = pd.read_csv("weekly AFR cases by country as of 19 January 2025(in).csv")
#print(cases_df.head())

In [14]:
#cleaning cases data
# Select relevant columns
cases_df_model = cases_df[['country', 'week_end_date', 'total_confirmed_cases', 'new_confirmed_cases']].copy()
cases_df_model['week_end_date'] = pd.to_datetime(cases_df_model['week_end_date'])

cases_df_model = cases_df_model.sort_values(by=['country', 'week_end_date'], ascending=[True, True])

# Create lag feature for new weekly cases
cases_df_model['previous_week_cases'] = cases_df_model.groupby('country', group_keys=False)['total_confirmed_cases'].shift(1)

# Update column safely without using inplace=True
cases_df_model['previous_week_cases'] = cases_df_model['previous_week_cases'].fillna(0)

cases_df_model['month'] = cases_df_model['week_end_date'].dt.month  # seasonality effect

#cases_df_model.iloc[170:191]

In [ ]:
healthcare_df = pd.read_csv("Healthcare expenditure data as percentage of GDP (%).csv")
healthcare_df.head()

In [16]:
warnings.filterwarnings("ignore")  # suppress warnings

population_df = pd.read_csv("f4b22788-e8b8-400a-b828-fb47aa226b25_Series - Metadata.csv")
population_df.head()

# Filter population data to include only the countries in 
# cases data
countries_in_cases = cases_df_model['country'].unique()
population_df_filtered = population_df[population_df['Country Name'].isin(countries_in_cases)]
population_df_filtered = population_df_filtered.iloc[:, 2:]
population_df_filtered = population_df_filtered.drop(columns=['Country Code'])
population_df_filtered.rename(columns={'Country Name':'country'}, inplace=True)
population_df_filtered.columns = population_df_filtered.columns.str.split(' ').str[0]

population_df_filtered.head()

year_columns = population_df_filtered.columns[1:].astype(int)
latest_year = np.int64(2023)

# Add empty columns for forecasts
population_df_filtered['2024'] = None
population_df_filtered['2025'] = None

# Loop over each country
for index, row in population_df_filtered.iterrows():
    populations = row[1:len(year_columns)+1].values.astype(float)
    
    # Build time series
    population_series = pd.Series(populations, index=year_columns)
    
    try:
        # Fit fixed ARIMA(1,1,1) model
        model = ARIMA(population_series, order=(1, 1, 1))
        model_fit = model.fit()
        
        # Forecast next 2 years
        forecast = model_fit.forecast(steps=2)
        
        # Store forecasts
        population_df_filtered.at[index, '2024'] = round(forecast.iloc[0])
        population_df_filtered.at[index, '2025'] = round(forecast.iloc[1])
        
    except Exception as e:
        print(f"Failed for {row['Country']}: {e}")
        population_df_filtered.at[index, '2024'] = None
        population_df_filtered.at[index, '2025'] = None

population_df_filtered.head()

,country,1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
5,Angola,11626360,12023529,12423712,12827135,13249764,13699778,14170973,14660413,15159370,...,29183070,30234839,31297155,32375632,33451132,34532429,35635029,36749906,37509312,38268707
20,Benin,5281479,5446932,5617844,5872706,6096506,6226773,6391858,6583365,6789489,...,11697842,12039780,12383347,12726755,13070169,13413417,13759501,14111034,14375550,14640062
31,Burundi,5587052,5703857,5858407,5676843,5713854,6066316,6070506,6070042,6187108,...,11239451,11506762,11859446,12255336,12617036,12965481,13321097,13689450,13933055,14176658
34,Cameroon,11331821,11667180,12006353,12353131,12704903,13058516,13414757,13775214,14144860,...,23454161,24128601,24806383,25506095,26210558,26915758,27632771,28372687,28889506,29406307
37,Central African Republic,2871910,2963191,3058831,3157839,3257954,3348052,3435965,3531700,3628827,...,4713663,4793511,4878657,4944703,5026628,5112100,5098039,5152421,5192241,5230872


In [17]:
#testing cases data

# Step 1: Have a data frame with relevant factors
df = cases_df_model.copy()

# Convert week_end_date to numerical format
df['week_end_date_num'] = df['week_end_date'].map(pd.Timestamp.toordinal)

# Step 2: Define features (x) and target (y)
x = df[['week_end_date_num', 'total_confirmed_cases','previous_week_cases', 'new_confirmed_cases', 'month']]
y = df['new_confirmed_cases']

# Step 3: Split data into training (80%) and testing (20%)
n = 10
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=n)

# Step 4: Train the Random Forest Model
rf = RandomForestRegressor(n_estimators=100, random_state=n)
rf.fit(x_train, y_train)

# Step 5: Compute Feature Importance
feature_importance = pd.DataFrame({
    'Feature': x.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("Feature Importance:")
print(feature_importance)

# Step 6: Make Predictions on Test Data
y_pred = rf.predict(x_test)

# Step 7: Evaluate Model
mae = mean_absolute_error(y_test, y_pred)
print(f'Mean Absolute Error: {mae:.2f}')

# Step 8: Predict Future (e.g. 2025) Monkeypox Cases
# function to predict
def predict_cases_for_date(last_known_date, total_cases_so_far, last_week_cases, future_date):
    last_known_date = pd.to_datetime(last_known_date)
    future_date = pd.to_datetime(future_date)
    
    if future_date <= last_known_date:
        print("Error: Future date must be after the last known date.")
        return None
    
    num_weeks = (future_date - last_known_date).days // 7
    
    predicted_total_cases = total_cases_so_far
    previous_week_cases = last_week_cases
    current_date = last_known_date

    for i in range(num_weeks):
        current_date_num = current_date.toordinal()
        month = current_date.month
        new_confirmed_cases = predicted_total_cases - previous_week_cases
        
        new_data = pd.DataFrame({
            'week_end_date_num': [current_date_num],
            'total_confirmed_cases': [predicted_total_cases],
            'previous_week_cases': [previous_week_cases],
            'new_confirmed_cases': [new_confirmed_cases],
            'month': [month],
        })
        
        # Predict this week's total cases
        predicted_new_cases = rf.predict(new_data)[0]
        
        # Update for the next iteration
        previous_week_cases = predicted_total_cases
        predicted_total_cases += predicted_new_cases
        current_date += pd.Timedelta(weeks=1)
    
    print(f"Predicted Monkeypox Cases for {future_date.date()}: {predicted_total_cases:.0f}")

Feature Importance:
                 Feature  Importance
3    new_confirmed_cases    0.998718
1  total_confirmed_cases    0.000394
2    previous_week_cases    0.000363
4                  month    0.000283
0      week_end_date_num    0.000242
Mean Absolute Error: 0.28


In [18]:
predict_cases_for_date('2025-01-05', 3035, 2946, '2025-01-12')

Predicted Monkeypox Cases for 2025-01-12: 3126


In [ ]:
# Step 1: Have a data frame with data of the different factors we want to look at
# e.g. df = pd.DataFrame()

# (Step 1.5: Could create lag features for previous year, for the model to use previous year data as another factor)
# e.g. df['Lag_Healthcare_Expenditure'] = df['Healthcare_Expenditure'].shift(1)

# Step 2: Define factors (x) and target (y)
# e.g. x = df[['Year', 'Lag_Healthcare_Expenditure', 'Population_Density']]
#      y = df['Monkeypox_Cases']

# Step 4: Split data into training and testing (e.g. 80% train, 20% test)
# n = 10
# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=n)
# random state to make sure results are repeatable

# Step 5: Train the Random Forest Model
# e.g. rf = RandomForestRegressor(n_estimators=100, random_state=n)
# rf.fit(x_train, y_train)

# Step 6: Compute Feature Importance
# feature_importance = pd.DataFrame({
#     'Feature': x.columns,
#     'Importance': rf.feature_importances_
# }).sort_values(by='Importance', ascending=False)

# Display Feature Importance
# print("Feature Importance:")
# print(feature_importance)

# Step 7: Make Predictions
# y_pred = rf.predict(x_test)

# Step 8: Evaluate the Model
# mae = mean_absolute_error(y_test, y_pred)
# print(f'Mean Absolute Error: {mae:.2f}')

# Step 9: Predict Future (e.g. 2025) Monkeypox Cases
# e.g. new_data = pd.DataFrame({
#     'Year': [2025],
#     'Lag_Healthcare_Expenditure': [5000],  # Previous year's value (2024)
#     'Lag_Population_Density': [150],  # Previous year's value (2024)
# })

# prediction_2025 = rf.predict(new_data)
# print(f'Predicted Monkeypox Cases for 2025: {prediction_2025[0]:.0f}')